# PM01 loco-manipulation: DeepMimic and AMP walkthrough

This CPU-only notebook checks the paper-to-code contracts behind the Isaac Lab tasks. DeepMimic combines an explicit imitation reward with a task reward ([arXiv:1804.02717, section 5.3](https://arxiv.org/abs/1804.02717)); AMP instead learns a style reward from state transitions and combines it with the task reward ([arXiv:2104.02180, sections 5--6](https://arxiv.org/abs/2104.02180)). The heavy-object ranges and randomization are adapted from [arXiv:2310.03191](https://arxiv.org/abs/2310.03191).

The PM01 asset has no wrist or gripper degrees of freedom, so the implemented tasks slide a tabletop object and push a heavy floor crate; they do not reproduce grasping or carrying.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import torch

repo = Path.cwd()
feature_path = repo / 'source/Gurukul/Gurukul/tasks/manager_based/beyondmimic/amp_features.py'
spec = spec_from_file_location('pm01_amp_features', feature_path)
features = module_from_spec(spec)
spec.loader.exec_module(features)

batch, joints, bodies = 4, 24, 13
zeros_j = torch.zeros(batch, joints)
root_pos = torch.tensor([[0.0, 0.0, 0.82]]).repeat(batch, 1)
root_quat = torch.tensor([[1.0, 0.0, 0.0, 0.0]]).repeat(batch, 1)
amp_state = features.build_amp_observation(
    zeros_j, zeros_j, root_pos, root_quat,
    torch.zeros(batch, 3), torch.zeros(batch, 3), torch.zeros(batch, bodies, 3),
)
expected = 2 * joints + 1 + 6 + 3 + 3 + 3 * bodies
assert amp_state.shape == (batch, expected)
print({'one_frame_features': expected, 'two_frame_transition': 2 * expected})

The AMP discriminator input includes joint position and velocity, root height and orientation, root linear and angular velocity, and root-relative tracked-body positions. It deliberately excludes reference phase, object state, and the goal: those belong to the task MDP, while the prior stays task-agnostic. Two frames form the transition used by the discriminator.

In [ ]:
# Paper-level reward composition checks. Actual task terms are computed by Isaac Lab.
task_reward = torch.tensor([0.2, 0.8])
explicit_imitation = torch.tensor([0.9, 0.6])
discriminator_score = torch.tensor([1.0, 0.0])

deepmimic_total = explicit_imitation + task_reward
amp_style = torch.clamp(1.0 - 0.25 * (discriminator_score - 1.0).square(), min=0.0)
amp_total = 0.5 * task_reward + 0.5 * amp_style
assert torch.all((0.0 <= amp_style) & (amp_style <= 1.0))
print({'deepmimic': deepmimic_total.tolist(), 'amp': amp_total.tolist()})

## What remains simulator-dependent

This notebook validates tensor and reward contracts only. Scene construction, contact behavior, task registration, and learning curves require Isaac Sim. AMP cannot infer manipulation contacts absent from the dance/walking archives; collecting PM01 object-interaction motion would be the next data improvement.